In [ ]:
# ============================================================================
# CELL 1: Install Packages
# ============================================================================
!pip install torch transformers==4.47.1 accelerate==0.26.1 scikit-learn numpy pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.9/270.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 75.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: accelerate
    Found existing installation: 

In [ ]:
# ============================================================================
# CELL 2: Imports
# ============================================================================
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import random
import re
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

Using device: cuda
GPU: Tesla T4
GPU Memory: 14.56 GB


In [ ]:
# ============================================================================
# CELL 3: Load Data
# ============================================================================
train_path = "train_data (1).json"
test_path = "test_data_subtask_1 (1).json"

with open(train_path, 'r') as f:
    train_data = json.load(f)

with open(test_path, 'r') as f:
    test_data = json.load(f)

train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"\nValidity distribution:\n{train_df['validity'].value_counts()}")
print(f"\nPlausibility distribution:\n{train_df['plausibility'].value_counts()}")

# Analyze 4 conditions
conditions = train_df.groupby(['plausibility', 'validity']).size()
print(f"\n4-Condition Distribution (Original):")
print(conditions)

Training samples: 960
Test samples: 191

Validity distribution:
validity
False    480
True     480
Name: count, dtype: int64

Plausibility distribution:
plausibility
False    486
True     474
Name: count, dtype: int64

4-Condition Distribution (Original):
plausibility  validity
False         False       246
              True        240
True          False       234
              True        240
dtype: int64


In [ ]:
# ============================================================================
# CELL 4: PROPER Train/Val Split BEFORE Augmentation
# ============================================================================
print("\n" + "="*80)
print("Step 1: Split ORIGINAL Data First (Prevent Data Leakage)")
print("="*80)

# Create condition column for stratification
train_df['condition'] = (
    train_df['plausibility'].astype(str) + '_' +
    train_df['validity'].astype(str)
)

# Split ORIGINAL data first
train_indices, val_indices = train_test_split(
    range(len(train_df)),
    test_size=0.15,
    random_state=42,
    stratify=train_df['condition'].values
)

train_df_original = train_df.iloc[train_indices].copy()
val_df_original = train_df.iloc[val_indices].copy()

print(f"\nOriginal data split:")
print(f"  Training: {len(train_df_original)}")
print(f"  Validation: {len(val_df_original)}")
print(f"\n CRITICAL: Augmentation will ONLY be applied to training data")
print(f"   Validation stays ORIGINAL to test true generalization")


Step 1: Split ORIGINAL Data First (Prevent Data Leakage)

Original data split:
  Training: 816
  Validation: 144

 CRITICAL: Augmentation will ONLY be applied to training data
   Validation stays ORIGINAL to test true generalization


In [ ]:
# ============================================================================
# CELL 5: Template-Based Augmentation (TRAINING ONLY)
# ============================================================================
print("\n" + "="*80)
print("Step 2: Augment ONLY Training Data")
print("="*80)

class TemplateAugmenter:
    """Generate variations that preserve logical structure"""

    def __init__(self):
        self.entity_pools = {
            'vehicles': ['car', 'bicycle', 'motorcycle', 'truck', 'bus', 'train', 'airplane', 'boat'],
            'buildings': ['house', 'building', 'tower', 'castle', 'barn', 'shed', 'cabin', 'mansion'],
            'animals': ['dog', 'cat', 'horse', 'bird', 'fish', 'rabbit', 'lion', 'tiger'],
            'people': ['person', 'human', 'individual', 'citizen', 'student', 'teacher', 'worker', 'doctor'],
            'objects': ['book', 'table', 'chair', 'tool', 'device', 'machine', 'instrument', 'gadget'],
            'food': ['fruit', 'vegetable', 'meat', 'drink', 'meal', 'snack', 'dish', 'dessert'],
            'nature': ['tree', 'plant', 'flower', 'rock', 'mountain', 'river', 'ocean', 'forest'],
        }

    def identify_entities(self, text):
        """Extract main entities"""
        entities = []
        text_lower = text.lower()

        for category, entity_list in self.entity_pools.items():
            for entity in entity_list:
                if entity in text_lower:
                    entities.append((entity, category))

        return list(set(entities))[:2]  # Max 2 entities

    def replace_entities(self, text, old_entities, new_entities):
        """Case-preserving replacement"""
        result = text

        for (old, _), (new, _) in zip(old_entities, new_entities):
            result = re.sub(re.escape(old), new, result, flags=re.IGNORECASE)
            result = re.sub(re.escape(old.capitalize()), new.capitalize(), result)

        return result

    def generate_variations(self, text, validity, plausibility, n_variations=2):
        """Generate n variations with different entities"""
        variations = []
        entities = self.identify_entities(text)

        if not entities:
            return variations

        for i in range(n_variations):
            new_entities = []

            for old_entity, old_category in entities:
                # Pick new entity from same category
                available = [e for e in self.entity_pools[old_category] if e != old_entity]
                if available:
                    new_entity = random.choice(available)
                    new_entities.append((new_entity, old_category))
                else:
                    new_entities.append((old_entity, old_category))

            if len(new_entities) == len(entities):
                new_text = self.replace_entities(text, entities, new_entities)

                if new_text != text:
                    variations.append({
                        'syllogism': new_text,
                        'validity': validity,
                        'plausibility': plausibility
                    })

        return variations

# Augment ONLY training data
augmenter = TemplateAugmenter()
augmented_train = []
variation_count = 0

for idx, row in tqdm(train_df_original.iterrows(), total=len(train_df_original), desc="Augmenting training"):
    # Add original
    augmented_train.append(row.to_dict())

    # Generate 2 variations
    variations = augmenter.generate_variations(
        row['syllogism'],
        row['validity'],
        row['plausibility'],
        n_variations=2
    )

    augmented_train.extend(variations)
    variation_count += len(variations)

train_df_augmented = pd.DataFrame(augmented_train)
train_df_augmented = train_df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n✓ Training data augmentation:")
print(f"  Original training: {len(train_df_original)}")
print(f"  Generated variations: {variation_count}")
print(f"  Total training: {len(train_df_augmented)}")
print(f"  Expansion: {len(train_df_augmented)/len(train_df_original):.2f}x")

print(f"\n✓ Validation data (UNCHANGED):")
print(f"  Validation samples: {len(val_df_original)}")
print(f"  Status: NO AUGMENTATION (true generalization test)")


Step 2: Augment ONLY Training Data


Augmenting training: 100%|██████████| 816/816 [00:00<00:00, 1711.88it/s]


✓ Training data augmentation:
  Original training: 816
  Generated variations: 1224
  Total training: 2040
  Expansion: 2.50x

✓ Validation data (UNCHANGED):
  Validation samples: 144
  Status: NO AUGMENTATION (true generalization test)


In [ ]:
# ============================================================================
# CELL 6: Oversample Implausible in Training Only
# ============================================================================
print("\n" + "="*80)
print("Step 3: Oversample Implausible (Training Only)")
print("="*80)

# Separate by plausibility in augmented training data
plausible_train = train_df_augmented[train_df_augmented['plausibility'] == True]
implausible_train = train_df_augmented[train_df_augmented['plausibility'] == False]

print(f"\nBefore oversampling:")
print(f"  Plausible: {len(plausible_train)}")
print(f"  Implausible: {len(implausible_train)}")

# Oversample implausible 2x
implausible_oversampled = pd.concat([implausible_train, implausible_train], ignore_index=True)

# Combine
train_df_final = pd.concat([plausible_train, implausible_oversampled], ignore_index=True)
train_df_final = train_df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nAfter oversampling:")
print(f"  Total training: {len(train_df_final)}")

# Add dummy counterfactuals
train_df_final['counterfactual'] = train_df_final['syllogism']
val_df_original['counterfactual'] = val_df_original['syllogism']

# Final data
train_df = train_df_final
val_df = val_df_original

conditions_train = train_df.groupby(['plausibility', 'validity']).size()
conditions_val = val_df.groupby(['plausibility', 'validity']).size()

print(f"\nFinal 4-Condition Distribution:")
print(f"  TRAINING (augmented + oversampled):")
for cond, count in conditions_train.items():
    print(f"    {cond}: {count}")
print(f"  VALIDATION (original, no augmentation):")
for cond, count in conditions_val.items():
    print(f"    {cond}: {count}")

print(f"\n KEY: Validation = original data → Tests true generalization")


Step 3: Oversample Implausible (Training Only)

Before oversampling:
  Plausible: 1035
  Implausible: 1005

After oversampling:
  Total training: 3045

Final 4-Condition Distribution:
  TRAINING (augmented + oversampled):
    (False, False): 1054
    (False, True): 956
    (True, False): 529
    (True, True): 506
  VALIDATION (original, no augmentation):
    (False, False): 37
    (False, True): 36
    (True, False): 35
    (True, True): 36

 KEY: Validation = original data → Tests true generalization


In [ ]:
# ============================================================================
# CELL 7: Dataset Class
# ============================================================================
class SyllogismDataset(Dataset):
    def __init__(self, texts, counterfactuals, validity_labels, plausibility_labels, tokenizer, max_length=512):
        self.texts = texts
        self.counterfactuals = counterfactuals
        self.validity_labels = validity_labels
        self.plausibility_labels = plausibility_labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded_orig = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        encoded_cf = self.tokenizer(
            self.counterfactuals[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids_orig': encoded_orig['input_ids'].squeeze(0),
            'attention_mask_orig': encoded_orig['attention_mask'].squeeze(0),
            'input_ids_cf': encoded_cf['input_ids'].squeeze(0),
            'attention_mask_cf': encoded_cf['attention_mask'].squeeze(0),
            'validity_label': torch.tensor(self.validity_labels[idx], dtype=torch.long),
            'plausibility_label': torch.tensor(self.plausibility_labels[idx], dtype=torch.long)
        }

In [ ]:
# ============================================================================
# CELL 8: Focal Loss
# ============================================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, inputs, targets):
     inputs = inputs.float()  # ensure float32
     ce_loss = F.cross_entropy(inputs, targets, reduction='none')
     ce_loss = torch.clamp(ce_loss, max=100.0)  # prevent exp overflow
     p_t = torch.exp(-ce_loss)
     focal_loss = (1 - p_t) ** self.gamma * ce_loss

     if self.alpha is not None:
        alpha_t = self.alpha[targets]
        focal_loss = alpha_t * focal_loss

     if self.reduction == 'mean':
        return focal_loss.mean()
     elif self.reduction == 'sum':
        return focal_loss.sum()
     else:
        return focal_loss

In [ ]:
# ============================================================================
# CELL 9: Gradient Reversal Layer
# ============================================================================
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_, None

class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_=1.0):
        super(GradientReversalLayer, self).__init__()
        self.lambda_ = lambda_

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_)

In [ ]:
# ============================================================================
# CELL 10: MLA-CI Model
# ============================================================================
class MLACIModel(nn.Module):
    def __init__(self, model_name, lambda_adv=1.0, dropout=0.2):
        super(MLACIModel, self).__init__()
        self.encoder = AutoModel.from_pretrained(
              model_name,
              output_hidden_states=True,
             low_cpu_mem_usage=False  # Force eager loading, disables accelerate's lazy materialization
             )
        self.encoder = self.encoder.float()  # Cast AFTER loading, not during

        hidden_size = self.encoder.config.hidden_size
        self.combined_size = hidden_size * 3
        self.grl = GradientReversalLayer(lambda_=lambda_adv)

        self.validity_classifier = nn.Sequential(
            nn.Linear(self.combined_size, 768),
            nn.LayerNorm(768),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2)
        )

        self.plausibility_adversary = nn.Sequential(
            nn.Linear(self.combined_size, 768),
            nn.LayerNorm(768),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2)
        )

    def extract_multi_layer_features(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.hidden_states

        layer_2 = hidden_states[2][:, 0, :]
        layer_6 = hidden_states[6][:, 0, :]
        layer_neg2 = hidden_states[-2][:, 0, :]

        combined = torch.cat([layer_2, layer_6, layer_neg2], dim=1)
        return combined.float()

    def forward(self, input_ids, attention_mask, return_features=False):
        features = self.extract_multi_layer_features(input_ids, attention_mask)

        validity_logits = self.validity_classifier(features)

        reversed_features = self.grl(features)
        plausibility_logits = self.plausibility_adversary(reversed_features)

        if return_features:
            return validity_logits, plausibility_logits, features
        return validity_logits, plausibility_logits

In [ ]:
# ============================================================================
# CELL 11: Data Preparation (Using Pre-Split Data)
# ============================================================================
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Use the already-split data (NO SECOND SPLIT!)
# train_df = augmented + oversampled training data
# val_df = original validation data (no augmentation)

print(f"\n Using Pre-Split Data:")
print(f"  Training: {len(train_df)} samples (augmented + oversampled)")
print(f"  Validation: {len(val_df)} samples (original, no augmentation)")

# Prepare training data
train_texts = train_df['syllogism'].tolist()
train_cf = train_df['counterfactual'].tolist()
train_validity = train_df['validity'].astype(int).values
train_plausibility = train_df['plausibility'].astype(int).values

# Prepare validation data
val_texts = val_df['syllogism'].tolist()
val_cf = val_df['counterfactual'].tolist()
val_validity = val_df['validity'].astype(int).values
val_plausibility = val_df['plausibility'].astype(int).values

# Create datasets
train_dataset = SyllogismDataset(train_texts, train_cf, train_validity, train_plausibility, tokenizer)
val_dataset = SyllogismDataset(val_texts, val_cf, val_validity, val_plausibility, tokenizer)

# Create dataloaders
BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nDataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Batch size: {BATCH_SIZE}")

# Initialize model
model = MLACIModel(MODEL_NAME, lambda_adv=1.0, dropout=0.2)
model = model.to(device)

print(f"  Model parameters: {sum(p.numel() for p in model.parameters()):,}")

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]


 Using Pre-Split Data:
  Training: 3045 samples (augmented + oversampled)
  Validation: 144 samples (original, no augmentation)

DataLoaders created:
  Train batches: 762
  Val batches: 36
  Batch size: 4


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

  Model parameters: 187,769,860


In [ ]:
import json
import copy
import gc

# ============================================================================
# HELPER: Modified model for single-layer ablation
# ============================================================================
class MLACIModel_SingleLayer(nn.Module):
    """MLA-CI but using ONLY the final [CLS] layer (no multi-layer)"""
    def __init__(self, model_name, lambda_adv=1.0, dropout=0.2):
        super(MLACIModel_SingleLayer, self).__init__()
        self.encoder = AutoModel.from_pretrained(
            model_name,
            output_hidden_states=True,
            low_cpu_mem_usage=False
        )
        self.encoder = self.encoder.float()
        hidden_size = self.encoder.config.hidden_size
        self.combined_size = hidden_size  # Only 1 layer = 768, NOT 768*3

        self.grl = GradientReversalLayer(lambda_=lambda_adv)

        # Same classifier architecture but input is 768 instead of 2304
        self.validity_classifier = nn.Sequential(
            nn.Linear(self.combined_size, 256),  # Smaller first layer since input is smaller
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2)
        )

        self.plausibility_adversary = nn.Sequential(
            nn.Linear(self.combined_size, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2)
        )

    def extract_multi_layer_features(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.hidden_states
        # Use ONLY the last layer [CLS]
        combined = hidden_states[-1][:, 0, :]
        return combined.float()

    def forward(self, input_ids, attention_mask, return_features=False):
        features = self.extract_multi_layer_features(input_ids, attention_mask)
        validity_logits = self.validity_classifier(features)
        reversed_features = self.grl(features)
        plausibility_logits = self.plausibility_adversary(reversed_features)
        if return_features:
            return validity_logits, plausibility_logits, features
        return validity_logits, plausibility_logits


# ============================================================================
# TRAINING + EVALUATION FUNCTION (reusable across all ablations)
# ============================================================================
def run_ablation_experiment(
    experiment_name,
    train_df_exp,
    val_df_exp,
    model_class=MLACIModel,
    lambda_adv=1.0,
    use_focal=True,
    focal_gamma=2.0,
    epochs=4,
    lr=2e-5,
    batch_size=4,
    accumulation_steps=4,
):
    """Run one full training experiment and return results."""

    print(f"\n{'#'*80}")
    print(f"# ABLATION: {experiment_name}")
    print(f"{'#'*80}")
    print(f"  Train samples: {len(train_df_exp)}")
    print(f"  Val samples:   {len(val_df_exp)}")
    print(f"  Lambda adv:    {lambda_adv}")
    print(f"  Focal loss:    {use_focal} (gamma={focal_gamma})")
    print(f"  Model class:   {model_class.__name__}")

    # Ensure counterfactual column exists
    if 'counterfactual' not in train_df_exp.columns:
        train_df_exp = train_df_exp.copy()
        train_df_exp['counterfactual'] = train_df_exp['syllogism']
    if 'counterfactual' not in val_df_exp.columns:
        val_df_exp = val_df_exp.copy()
        val_df_exp['counterfactual'] = val_df_exp['syllogism']

    # Prepare data
    tr_texts = train_df_exp['syllogism'].tolist()
    tr_cf = train_df_exp['counterfactual'].tolist()
    tr_validity = train_df_exp['validity'].astype(int).values
    tr_plausibility = train_df_exp['plausibility'].astype(int).values

    v_texts = val_df_exp['syllogism'].tolist()
    v_cf = val_df_exp['counterfactual'].tolist()
    v_validity = val_df_exp['validity'].astype(int).values
    v_plausibility = val_df_exp['plausibility'].astype(int).values

    # Datasets & loaders
    tr_dataset = SyllogismDataset(tr_texts, tr_cf, tr_validity, tr_plausibility, tokenizer)
    v_dataset = SyllogismDataset(v_texts, v_cf, v_validity, v_plausibility, tokenizer)
    tr_loader = DataLoader(tr_dataset, batch_size=batch_size, shuffle=True)
    v_loader = DataLoader(v_dataset, batch_size=batch_size, shuffle=False)

    # Model
    exp_model = model_class(MODEL_NAME, lambda_adv=lambda_adv, dropout=0.2)
    exp_model = exp_model.to(device)

    # Loss functions
    if use_focal:
        val_criterion = FocalLoss(gamma=focal_gamma, alpha=None)
    else:
        val_criterion = nn.CrossEntropyLoss()
    adv_criterion = nn.CrossEntropyLoss()

    # Optimizer & scheduler
    exp_optimizer = torch.optim.AdamW(exp_model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = (len(tr_loader) // accumulation_steps) * epochs
    exp_scheduler = get_linear_schedule_with_warmup(
        exp_optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    best_score = 0
    best_results = {}
    epoch_results = []

    for epoch in range(epochs):
        # --- TRAIN ---
        exp_model.train()
        total_loss = 0
        correct = 0
        total = 0
        exp_optimizer.zero_grad()

        for batch_idx, batch in enumerate(tqdm(tr_loader, desc=f"[{experiment_name}] Epoch {epoch+1}")):
            input_ids = batch['input_ids_orig'].to(device)
            attention_mask = batch['attention_mask_orig'].to(device)
            val_labels = batch['validity_label'].to(device)
            plaus_labels = batch['plausibility_label'].to(device)

            validity_logits, plausibility_logits = exp_model(input_ids, attention_mask)

            loss_val = val_criterion(validity_logits, val_labels)
            loss_plaus = adv_criterion(plausibility_logits, plaus_labels)
            loss = (loss_val + lambda_adv * loss_plaus) / accumulation_steps
            loss.backward()

            if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(tr_loader):
                torch.nn.utils.clip_grad_norm_(exp_model.parameters(), 1.0)
                exp_optimizer.step()
                exp_scheduler.step()
                exp_optimizer.zero_grad()

            total_loss += loss.item() * accumulation_steps
            preds = torch.argmax(validity_logits, dim=1)
            correct += (preds == val_labels).sum().item()
            total += val_labels.size(0)

        train_acc = correct / total

        # --- EVALUATE ---
        exp_model.eval()
        all_preds = []
        all_val = []
        all_plaus = []
        idx = 0

        with torch.no_grad():
            for batch in v_loader:
                input_ids = batch['input_ids_orig'].to(device)
                attention_mask = batch['attention_mask_orig'].to(device)
                val_labels = batch['validity_label'].to(device)
                bs = val_labels.size(0)

                validity_logits, _ = exp_model(input_ids, attention_mask)
                preds = torch.argmax(validity_logits, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_val.extend(val_labels.cpu().numpy())
                for i in range(bs):
                    all_plaus.append(v_plausibility[idx])
                    idx += 1

        all_preds = np.array(all_preds)
        all_val = np.array(all_val)
        all_plaus = np.array(all_plaus)

        # Overall accuracy
        val_acc = accuracy_score(all_val, all_preds)

        # Per-condition accuracy
        conditions = {}
        for plaus in [0, 1]:
            for valid in [0, 1]:
                mask = (all_plaus == plaus) & (all_val == valid)
                if mask.sum() > 0:
                    acc = accuracy_score(all_val[mask], all_preds[mask])
                    plaus_name = "Plausible" if plaus == 1 else "Implausible"
                    valid_name = "Valid" if valid == 1 else "Invalid"
                    conditions[f"{plaus_name}_{valid_name}"] = acc

        # Content effect
        intra_plaus = abs(conditions.get('Plausible_Valid', 0) - conditions.get('Plausible_Invalid', 0))
        intra_implaus = abs(conditions.get('Implausible_Valid', 0) - conditions.get('Implausible_Invalid', 0))
        intra_effect = (intra_plaus + intra_implaus) / 2

        cross_valid = abs(conditions.get('Plausible_Valid', 0) - conditions.get('Implausible_Valid', 0))
        cross_invalid = abs(conditions.get('Plausible_Invalid', 0) - conditions.get('Implausible_Invalid', 0))
        cross_effect = (cross_valid + cross_invalid) / 2

        val_ce = (intra_effect + cross_effect) / 2

        # Combined score (OFFICIAL FORMULA)
        combined = val_acc * 100 / (1 + np.log(1 + val_ce * 100))

        print(f"  Epoch {epoch+1}: Acc={val_acc*100:.2f}%, CE={val_ce*100:.2f}%, Score={combined:.2f}")

        epoch_results.append({
            'epoch': epoch + 1,
            'train_acc': round(train_acc * 100, 2),
            'val_acc': round(val_acc * 100, 2),
            'val_ce': round(val_ce * 100, 2),
            'combined_score': round(combined, 2),
            'conditions': {k: round(v * 100, 2) for k, v in conditions.items()}
        })

        if combined > best_score:
            best_score = combined
            best_results = {
                'experiment': experiment_name,
                'best_epoch': epoch + 1,
                'val_acc': round(val_acc * 100, 2),
                'val_ce': round(val_ce * 100, 2),
                'combined_score': round(combined, 2),
                'conditions': {k: round(v * 100, 2) for k, v in conditions.items()},
            }

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    best_results['all_epochs'] = epoch_results

    # Cleanup
    del exp_model, exp_optimizer, exp_scheduler, tr_loader, v_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"\n  ★ BEST for [{experiment_name}]: Epoch {best_results['best_epoch']}, "
          f"Acc={best_results['val_acc']}%, CE={best_results['val_ce']}%, "
          f"Score={best_results['combined_score']}")

    return best_results


# ============================================================================
# PREPARE DATA VARIANTS
# ============================================================================
# At this point in the notebook, you should have these variables from cells 1-6:
#   train_df_original  (816 samples, the 85% split of original data)
#   val_df_original    (144 samples, the 15% split — NEVER augmented)
#   train_df_augmented (2040 samples, after template augmentation)
#   train_df           (3045 samples, after augmentation + oversampling) — this is train_df_final

# IMPORTANT: We use val_df_original for ALL ablations (consistent evaluation)
val_df_abl = val_df_original.copy()

# Data variant 1: Full pipeline (augmented + oversampled) — this is 'train_df' from cell 6
full_train = train_df.copy()  # 3045 samples

# Data variant 2: No augmentation (original training + oversampled implausible)
no_aug_plausible = train_df_original[train_df_original['plausibility'] == True]
no_aug_implausible = train_df_original[train_df_original['plausibility'] == False]
no_aug_implausible_2x = pd.concat([no_aug_implausible, no_aug_implausible], ignore_index=True)
no_aug_train = pd.concat([no_aug_plausible, no_aug_implausible_2x], ignore_index=True)
no_aug_train = no_aug_train.sample(frac=1, random_state=42).reset_index(drop=True)

# Data variant 3: No oversampling (augmented but NOT oversampled)
no_oversample_train = train_df_augmented.copy()  # 2040 samples, no oversampling

print("="*60)
print("DATA VARIANTS FOR ABLATION")
print("="*60)
print(f"  Full pipeline (aug + oversample):  {len(full_train)} samples")
print(f"  No augmentation (orig + oversample): {len(no_aug_train)} samples")
print(f"  No oversampling (aug only):          {len(no_oversample_train)} samples")
print(f"  Validation (always original):        {len(val_df_abl)} samples")


# ============================================================================
# RUN ALL 6 ABLATION EXPERIMENTS
# ============================================================================
ABLATION_EPOCHS = 4  # Set to 2 to save time (best is usually epoch 2)
all_results = {}

# ── Experiment 0: FULL SYSTEM (baseline for comparison) ──
all_results['full'] = run_ablation_experiment(
    experiment_name="Full MLA-CI",
    train_df_exp=full_train,
    val_df_exp=val_df_abl,
    model_class=MLACIModel,
    lambda_adv=1.0,
    use_focal=True,
    focal_gamma=2.0,
    epochs=ABLATION_EPOCHS,
)

# ── Experiment 1: REMOVE AUGMENTATION ──
all_results['no_aug'] = run_ablation_experiment(
    experiment_name="− Augmentation",
    train_df_exp=no_aug_train,
    val_df_exp=val_df_abl,
    model_class=MLACIModel,
    lambda_adv=1.0,
    use_focal=True,
    focal_gamma=2.0,
    epochs=ABLATION_EPOCHS,
)

# ── Experiment 2: REMOVE ADVERSARIAL (λ=0) ──
all_results['no_adv'] = run_ablation_experiment(
    experiment_name="− Adversarial (λ=0)",
    train_df_exp=full_train,
    val_df_exp=val_df_abl,
    model_class=MLACIModel,
    lambda_adv=0.0,         # <-- KEY CHANGE
    use_focal=True,
    focal_gamma=2.0,
    epochs=ABLATION_EPOCHS,
)

# ── Experiment 3: REMOVE FOCAL LOSS (use standard CE) ──
all_results['no_focal'] = run_ablation_experiment(
    experiment_name="− Focal Loss",
    train_df_exp=full_train,
    val_df_exp=val_df_abl,
    model_class=MLACIModel,
    lambda_adv=1.0,
    use_focal=False,         # <-- KEY CHANGE
    focal_gamma=2.0,
    epochs=ABLATION_EPOCHS,
)

# ── Experiment 4: REMOVE OVERSAMPLING ──
all_results['no_oversample'] = run_ablation_experiment(
    experiment_name="− Oversampling",
    train_df_exp=no_oversample_train,
    val_df_exp=val_df_abl,
    model_class=MLACIModel,
    lambda_adv=1.0,
    use_focal=True,
    focal_gamma=2.0,
    epochs=ABLATION_EPOCHS,
)

# ── Experiment 5: REMOVE MULTI-LAYER (single layer only) ──
all_results['no_multilayer'] = run_ablation_experiment(
    experiment_name="− Multi-layer",
    train_df_exp=full_train,
    val_df_exp=val_df_abl,
    model_class=MLACIModel_SingleLayer,  # <-- KEY CHANGE
    lambda_adv=1.0,
    use_focal=True,
    focal_gamma=2.0,
    epochs=ABLATION_EPOCHS,
)


# ============================================================================
# PRINT FINAL SUMMARY TABLE (copy this directly into your paper)
# ============================================================================
print("\n" + "="*80)
print("ABLATION RESULTS SUMMARY")
print("="*80)
print(f"{'Configuration':<30} {'Acc (%)':<10} {'CE (%)':<10} {'Score':<10} {'Best Ep':<8}")
print("-"*68)

for key in ['full', 'no_aug', 'no_adv', 'no_focal', 'no_oversample', 'no_multilayer']:
    r = all_results[key]
    print(f"{r['experiment']:<30} {r['val_acc']:<10} {r['val_ce']:<10} {r['combined_score']:<10} {r['best_epoch']:<8}")

print("-"*68)
print()

# Compute deltas from full system
full_score = all_results['full']['combined_score']
full_acc = all_results['full']['val_acc']
full_ce = all_results['full']['val_ce']

print(f"{'Configuration':<30} {'Δ Acc':<10} {'Δ CE':<10} {'Δ Score':<10}")
print("-"*60)
for key in ['no_aug', 'no_adv', 'no_focal', 'no_oversample', 'no_multilayer']:
    r = all_results[key]
    d_acc = r['val_acc'] - full_acc
    d_ce = r['val_ce'] - full_ce  # positive means CE got worse
    d_score = r['combined_score'] - full_score
    print(f"{r['experiment']:<30} {d_acc:+.2f}{'':5} {d_ce:+.2f}{'':5} {d_score:+.2f}")

print()
print("NOTE: For CE, positive delta = worse (more content bias)")
print("      For Acc and Score, negative delta = worse")

# Per-condition breakdown
print("\n" + "="*80)
print("PER-CONDITION ACCURACY (%)")
print("="*80)
header = f"{'Config':<25} {'P-V':<8} {'P-I':<8} {'I-V':<8} {'I-I':<8} {'StdDev':<8}"
print(header)
print("-"*65)
for key in ['full', 'no_aug', 'no_adv', 'no_focal', 'no_oversample', 'no_multilayer']:
    r = all_results[key]
    c = r['conditions']
    pv = c.get('Plausible_Valid', 0)
    pi = c.get('Plausible_Invalid', 0)
    iv = c.get('Implausible_Valid', 0)
    ii = c.get('Implausible_Invalid', 0)
    std = np.std([pv, pi, iv, ii])
    print(f"{r['experiment']:<25} {pv:<8.1f} {pi:<8.1f} {iv:<8.1f} {ii:<8.1f} {std:<8.2f}")


# ============================================================================
# SAVE RESULTS TO FILE
# ============================================================================
with open('ablation_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"\n Results saved to ablation_results.json")
print(f" Copy the table above directly into your paper's Table 4")


# ============================================================================
# ALSO: Generate LaTeX table for the paper
# ============================================================================
print("\n" + "="*80)
print("LATEX TABLE (paste into your paper)")
print("="*80)
print(r"""
\begin{table}[t]
\centering
\small
\begin{tabular}{lccc}
\toprule
\textbf{Configuration} & \textbf{Acc (\%)} & \textbf{CE (\%)} & \textbf{Score} \\
\midrule""")

for key in ['full', 'no_aug', 'no_adv', 'no_focal', 'no_oversample', 'no_multilayer']:
    r = all_results[key]
    name_map = {
        'full': r'Full MLA-CI',
        'no_aug': r'$-$ Augmentation',
        'no_adv': r'$-$ Adversarial',
        'no_focal': r'$-$ Focal loss',
        'no_oversample': r'$-$ Oversampling',
        'no_multilayer': r'$-$ Multi-layer',
    }
    name = name_map[key]
    print(f"{name} & {r['val_acc']:.1f} & {r['val_ce']:.1f} & {r['combined_score']:.2f} \\\\")

print(r"""\midrule
v3 (no aug., test) & --- & --- & 26.00 \\
\bottomrule
\end{tabular}
\caption{Ablation results on validation. Each row removes one component from the full system.}
\label{tab:ablation}
\end{table}""")

DATA VARIANTS FOR ABLATION
  Full pipeline (aug + oversample):  3045 samples
  No augmentation (orig + oversample): 1229 samples
  No oversampling (aug only):          2040 samples
  Validation (always original):        144 samples

################################################################################
# ABLATION: Full MLA-CI
################################################################################
  Train samples: 3045
  Val samples:   144
  Lambda adv:    1.0
  Focal loss:    True (gamma=2.0)
  Model class:   MLACIModel


[Full MLA-CI] Epoch 1: 100%|██████████| 762/762 [06:56<00:00,  1.83it/s]


  Epoch 1: Acc=66.67%, CE=30.04%, Score=15.03


[Full MLA-CI] Epoch 2: 100%|██████████| 762/762 [06:53<00:00,  1.84it/s]


  Epoch 2: Acc=81.94%, CE=12.90%, Score=22.56


...

  Epoch 2: Acc=69.44%, CE=24.92%, Score=16.32


[− Multi-layer] Epoch 3: 100%|██████████| 762/762 [07:12<00:00,  1.76it/s]


  Epoch 3: Acc=71.53%, CE=20.07%, Score=17.67


[− Multi-layer] Epoch 4: 100%|██████████| 762/762 [07:12<00:00,  1.76it/s]


  Epoch 4: Acc=72.92%, CE=23.53%, Score=17.36

  ★ BEST for [− Multi-layer]: Epoch 3, Acc=71.53%, CE=20.07%, Score=17.67

ABLATION RESULTS SUMMARY
Configuration                  Acc (%)    CE (%)     Score      Best Ep 
--------------------------------------------------------------------
Full MLA-CI                    83.33      8.33       25.77      4       
− Augmentation                 72.92      14.44      19.51      4       
− Adversarial (λ=0)            93.06      4.2        35.12      2       
− Focal Loss                   90.28      4.44       33.51      4       
− Oversampling                 86.11      5.56       29.9       4       
− Multi-layer                  71.53      20.07      17.67      3       
--------------------------------------------------------------------

Configuration                  Δ Acc      Δ CE       Δ Score   
------------------------------------------------------------
− Augmentation                 -10.41      +6.11      -6.26
− Adversarial (λ=0